# GCP Dataset EDA

Exploratory data analysis for GCP keypoint localization and shape classification.

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT.parent / 'data'
TRAIN_ROOT = DATA_ROOT / 'train_dataset'
LABELS_PATH = TRAIN_ROOT / 'gcp_marks.json'

with LABELS_PATH.open('r', encoding='utf-8') as f:
    labels = json.load(f)

rows = []
for relative_path, annotation in labels.items():
    rows.append({
        'relative_path': relative_path,
        'x': annotation.get('mark', {}).get('x'),
        'y': annotation.get('mark', {}).get('y'),
        'verified_shape': annotation.get('verified_shape', 'MISSING')
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
print('Samples:', len(df))
print(df['verified_shape'].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['verified_shape'].value_counts(dropna=False).plot(kind='bar', ax=axes[0], title='Class Distribution')
axes[0].set_xlabel('Shape')
axes[0].set_ylabel('Count')
axes[1].scatter(df['x'], df['y'], s=10, alpha=0.4)
axes[1].set_title('Keypoint Distribution')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].invert_yaxis()
plt.tight_layout()

In [ ]:
sample_df = df[df['verified_shape'] != 'MISSING'].sample(6, random_state=42).reset_index(drop=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (_, row) in zip(axes.ravel(), sample_df.iterrows()):
    image_path = TRAIN_ROOT / Path(row['relative_path'])
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    cv2.circle(image, (int(row['x']), int(row['y'])), 18, (255, 0, 0), -1)
    ax.imshow(image)
    ax.set_title(f"{row['verified_shape']}\n{row['relative_path']}", fontsize=9)
    ax.axis('off')

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['x'].hist(ax=axes[0], bins=30)
axes[0].set_title('X Coordinate Distribution')
df['y'].hist(ax=axes[1], bins=30)
axes[1].set_title('Y Coordinate Distribution')
plt.tight_layout()